In [1]:
import pandas as pd
import numpy as np
import hashlib
import unicodedata

In [ ]:
fighter_tott = pd.read_csv(r'C:\Users\edude\OneDrive\Área de Trabalho\Códigos\Faculdade\TCC\LabA\base_dados\processed_bronze\1.scrape_ufc_stats\fighter_tott.csv')
raw_fighter_details = pd.read_csv(r'C:\Users\edude\OneDrive\Área de Trabalho\Códigos\Faculdade\TCC\LabA\base_dados\processed_bronze\3.ufc_datalab\raw_fighter_details.csv')
ufc_dataset = pd.read_csv(r'C:\Users\edude\OneDrive\Área de Trabalho\Códigos\Faculdade\TCC\LabA\base_dados\processed_bronze\ufc_dataset\fighter_details.csv')

In [3]:
def normalize_name_col(s):
    return (
        s.astype(str)
        .str.upper()
        .str.strip()
        .str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("utf-8")
        .str.replace(r"\s+", " ", regex=True)
    )


def make_internal_id(key):
    return hashlib.md5(key.encode("utf-8")).hexdigest()[:16]


def first_valid(series):
    valid = series.dropna()
    valid = valid[valid.astype(str).str.strip() != ""]
    return valid.iloc[0] if len(valid) > 0 else np.nan


def extract_ufcstats_id(url_series):
    return url_series.astype(str).str.extract(r"fighter-details/([^/]+)")[0]

In [4]:
tott = fighter_tott.copy()

tott["fighter_key"] = normalize_name_col(tott["fighter_name"])
tott["fighter_id"] = extract_ufcstats_id(tott["url"])

tott_std = pd.DataFrame({
    "fighter_id": tott["fighter_id"],
    "fighter_key": tott["fighter_key"],
    "fighter_name": tott["fighter_name"],
    "nick_name": np.nan,

    "wins": np.nan,
    "losses": np.nan,
    "draws": np.nan,

    "height": tott["height"],
    "weight": tott["weight"],
    "reach": tott["reach"],
    "stance": tott["stance"],
    "date_of_birth": tott["date_of_birth"],

    "height_cm": tott["height_cm"],
    "weight_lbs": tott["weight_lbs"],
    "reach_inches": tott["reach_inches"],
    "reach_cm": tott["reach_cm"],

    "url": tott["url"],

    "slpm": np.nan,
    "str_acc": np.nan,
    "str_acc_num": np.nan,
    "sapm": np.nan,
    "str_def": np.nan,
    "takedowns_avg": np.nan,
    "takedowns_acc": np.nan,
    "takedowns_acc_num": np.nan,
    "takedowns_def": np.nan,
    "sub_avg": np.nan,

    "source": "fighter_tott",
    "source_priority": 1
})

In [5]:
raw = raw_fighter_details.copy()

raw["fighter_key"] = normalize_name_col(raw["fighter_name"])

raw_std = pd.DataFrame({
    "fighter_id": np.nan,
    "fighter_key": raw["fighter_key"],
    "fighter_name": raw["fighter_name"],
    "nick_name": np.nan,

    "wins": np.nan,
    "losses": np.nan,
    "draws": np.nan,

    "height": raw["height"],
    "weight": raw["weight"],
    "reach": raw["reach"],
    "stance": raw["stance"],
    "date_of_birth": raw["date_of_birth"],

    "height_cm": np.nan,
    "weight_lbs": raw["weight_lbs"],
    "reach_inches": raw["reach_inches"],
    "reach_cm": raw["reach_inches"] * 2.54,

    "url": np.nan,

    "slpm": raw["slpm"],
    "str_acc": raw["str_acc"],
    "str_acc_num": raw["str_acc_num"],
    "sapm": raw["sapm"],
    "str_def": raw["str_def"],
    "takedowns_avg": raw["takedowns_avg"],
    "takedowns_acc": raw["takedowns_acc"],
    "takedowns_acc_num": raw["takedowns_acc_num"],
    "takedowns_def": raw["takedowns_def"],
    "sub_avg": raw["sub_avg"],

    "source": "raw_fighter_details",
    "source_priority": 2
})

In [6]:
ufc = ufc_dataset.copy()

ufc["fighter_key"] = normalize_name_col(ufc["name"])

ufc_std = pd.DataFrame({
    "fighter_id": ufc["id"],
    "fighter_key": ufc["fighter_key"],
    "fighter_name": ufc["name"],
    "nick_name": ufc["nick_name"],

    "wins": ufc["wins"],
    "losses": ufc["losses"],
    "draws": ufc["draws"],

    "height": np.nan,
    "weight": np.nan,
    "reach": np.nan,
    "stance": ufc["stance"],
    "date_of_birth": ufc["date_of_birth"],

    "height_cm": ufc["height"],
    "weight_lbs": ufc["weight"] * 2.20462,
    "reach_inches": ufc["reach"] / 2.54,
    "reach_cm": ufc["reach"],

    "url": np.nan,

    "slpm": ufc["splm"],
    "str_acc": ufc["str_acc"].astype(str) + "%",
    "str_acc_num": ufc["str_acc"],
    "sapm": ufc["sapm"],
    "str_def": ufc["str_def"].astype(str) + "%",
    "takedowns_avg": ufc["takedowns_avg"],
    "takedowns_acc": ufc["takedowns_avg_acc"].astype(str) + "%",
    "takedowns_acc_num": ufc["takedowns_avg_acc"],
    "takedowns_def": ufc["takedowns_def"],
    "sub_avg": ufc["sub_avg"],

    "source": "ufc_dataset",
    "source_priority": 3
})

In [7]:
fighters_all = pd.concat(
    [tott_std, raw_std, ufc_std],
    ignore_index=True
)

fighters_all = fighters_all.sort_values(
    by=["fighter_key", "source_priority"]
)

In [8]:
fighters_dim = (
    fighters_all
    .groupby("fighter_key", as_index=False)
    .agg({
        "fighter_id": first_valid,
        "fighter_name": first_valid,
        "nick_name": first_valid,

        "wins": first_valid,
        "losses": first_valid,
        "draws": first_valid,

        "height": first_valid,
        "weight": first_valid,
        "reach": first_valid,
        "stance": first_valid,
        "date_of_birth": first_valid,

        "height_cm": first_valid,
        "weight_lbs": first_valid,
        "reach_inches": first_valid,
        "reach_cm": first_valid,

        "url": first_valid,

        "slpm": first_valid,
        "str_acc": first_valid,
        "str_acc_num": first_valid,
        "sapm": first_valid,
        "str_def": first_valid,
        "takedowns_avg": first_valid,
        "takedowns_acc": first_valid,
        "takedowns_acc_num": first_valid,
        "takedowns_def": first_valid,
        "sub_avg": first_valid,

        "source": lambda x: ", ".join(sorted(set(x.dropna())))
    })
)

In [9]:
fighters_dim["fighter_id"] = fighters_dim["fighter_id"].combine_first(
    fighters_dim["fighter_key"].apply(make_internal_id)
)
fighters_dim = fighters_dim.rename(columns={
    "source": "source_match"
})

In [10]:
fighters_dim.shape

(4534, 28)

In [14]:
fighters_dim

fighters_dim[fighters_dim['fighter_name'].str.contains('Adam Fugitt', case=False, na=False)].to_dict()

{'fighter_key': {36: 'ADAM FUGITT'},
 'fighter_id': {36: 'a01a62132460a98d'},
 'fighter_name': {36: 'Adam Fugitt'},
 'nick_name': {36: nan},
 'wins': {36: 10.0},
 'losses': {36: 5.0},
 'draws': {36: 0.0},
 'height': {36: '6\' 1"'},
 'weight': {36: '170 lbs.'},
 'reach': {36: '77"'},
 'stance': {36: 'Southpaw'},
 'date_of_birth': {36: '1989-01-12'},
 'height_cm': {36: 185.42},
 'weight_lbs': {36: 170.0},
 'reach_inches': {36: 77.0},
 'reach_cm': {36: 195.58},
 'url': {36: 'http://ufcstats.com/fighter-details/a01a62132460a98d'},
 'slpm': {36: 4.59},
 'str_acc': {36: '46%'},
 'str_acc_num': {36: 46.0},
 'sapm': {36: 4.54},
 'str_def': {36: '51%'},
 'takedowns_avg': {36: 1.83},
 'takedowns_acc': {36: '25%'},
 'takedowns_acc_num': {36: 25.0},
 'takedowns_def': {36: 50},
 'sub_avg': {36: 0.0},
 'source_match': {36: 'fighter_tott, ufc_dataset'}}

In [22]:
fighters_dim.to_csv(r'C:\Users\edude\OneDrive\Área de Trabalho\Códigos\Faculdade\TCC\LabA\base_dados\processed_silver\dim_fighters.csv', sep=',', index=False)